In [45]:
import pandas as pd
import numpy as np
import plotly.express as px
import warnings
warnings.filterwarnings('ignore')
from sklearn.svm import OneClassSVM
from sklearn.preprocessing import StandardScaler, RobustScaler
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from sklearn.ensemble import IsolationForest
import math

In [1]:
import sys
import os

sys.path.append(os.path.abspath(".."))

from models.scaling import scale_features

from config import load_data, FEATURES, SKEWED, FRAUD_IDS
df, client_data = load_data()

##### Get data

In [2]:
# тільки активні клієнти
client_data = client_data[client_data['num_of_trn'] > 1]

##### Scaling

In [3]:
X_std = scale_features(scaler_type="standard", data=client_data, features=FEATURES, skewed=SKEWED)
X_rob = scale_features(scaler_type="robust", data=client_data, features=FEATURES, skewed=SKEWED)

#### One calss SVM

##### Standart Scaler

In [ ]:
check = client_data.copy()

for nu in [0.001, 0.002, 0.005, 0.01]:
    ocsvm = OneClassSVM(kernel="rbf", nu=nu, gamma="scale")
    pred = ocsvm.fit_predict(X_std)
    check[f"ocsvm_label_std_nu_{nu}"] = pred
    hit_rate_std = check[check['is_fraud']][f"ocsvm_label_std_nu_{nu}"].value_counts(normalize=True).get(-1, 0)
    print(f"One-Class SVM (StandardScaler) with nu={nu}: Anomaly Hit Rate = {hit_rate_std:.4f}")

One-Class SVM (StandardScaler) with nu=0.001: Anomaly Hit Rate = 0.1250  
One-Class SVM (StandardScaler) with nu=0.002: Anomaly Hit Rate = 0.2500  
One-Class SVM (StandardScaler) with nu=0.005: Anomaly Hit Rate = 0.5000  
One-Class SVM (StandardScaler) with nu=0.01: Anomaly Hit Rate = 0.7500  

In [ ]:
ocsvm = OneClassSVM(kernel="rbf", nu=0.01, gamma="scale")
pred = ocsvm.fit_predict(X_std)

client_data["ocsvm_label_std"] = pred

for pid in FRAUD_IDS:
    clusters = client_data.loc[client_data['person_id'] == pid, 'ocsvm_label_std'].tolist()
    print(f"person_id={pid} -> clusters={clusters}")


print()
client_data['ocsvm_label_std'].value_counts()

hit_rate_std = client_data[client_data['is_fraud']]['ocsvm_label_std'].value_counts(normalize=True).get(-1, 0)
print(f"Hit Rate (StandardScaler): {hit_rate_std:.2%}")

person_id=11968000 -> clusters=[-1]  
person_id=11970409 -> clusters=[-1]  
person_id=11726701 -> clusters=[1]  
person_id=14827913 -> clusters=[1]  
person_id=12411311 -> clusters=[-1]  
person_id=11098795 -> clusters=[]  
person_id=12412748 -> clusters=[-1]  
person_id=11812494 -> clusters=[-1]  
person_id=12396334 -> clusters=[-1]  
  
Hit Rate (StandardScaler): 75.00%  

In [ ]:
client_data["group"] = np.where(client_data["ocsvm_label_std"] == -1, "Anomaly_Std", "Normal")

client_data.loc[client_data["person_id"].isin(FRAUD_IDS), "group"] = "Known fraud"
group_order = ["Normal", "Anomaly_Std", "Known fraud"]

medians_wide = (
    client_data
    .groupby("group")[FEATURES]
    .median()
    .reindex(group_order)
    .reset_index()
)

medians_long = medians_wide.melt(
    id_vars="group",
    value_vars=FEATURES,
    var_name="feature",
    value_name="median_value"
)

fig = px.bar(
    medians_long,
    x="group",
    y="median_value",
    color='group',
    facet_col="feature",
    facet_col_wrap=4,
    category_orders={"group": group_order, "feature": FEATURES},
    title="Feature medians: Normal vs Anomaly vs Known fraud (client_data, standardized with StandardScaler)"
)

fig.update_xaxes(title=None, tickangle=0)
fig.update_yaxes(title="median value")
fig.for_each_annotation(lambda a: a.update(text=a.text.split("=")[-1])) 
fig.update_layout(height=320 * int(np.ceil(len(FEATURES)/4)), showlegend=False)
fig.update_yaxes(matches=None)

fig.show()

##### Robsut Scaler

In [ ]:
for nu in [0.001, 0.002, 0.005, 0.01]:
    ocsvm = OneClassSVM(kernel="rbf", nu=nu, gamma="scale")
    pred = ocsvm.fit_predict(X_rob)
    check[f"ocsvm_label_rob_nu_{nu}"] = pred
    hit_rate_rob = check[check['is_fraud']][f"ocsvm_label_rob_nu_{nu}"].value_counts(normalize=True).get(-1, 0)
    print(f"One-Class SVM (RobustScaler) with nu={nu}: Anomaly Hit Rate = {hit_rate_rob:.4f}")

One-Class SVM (RobustScaler) with nu=0.001: Anomaly Hit Rate = 0.1250  
One-Class SVM (RobustScaler) with nu=0.002: Anomaly Hit Rate = 0.3750  
One-Class SVM (RobustScaler) with nu=0.005: Anomaly Hit Rate = 0.6250  
One-Class SVM (RobustScaler) with nu=0.01: Anomaly Hit Rate = 0.8750  

In [ ]:
ocsvm = OneClassSVM(kernel="rbf", nu=0.01, gamma="scale")
pred = ocsvm.fit_predict(X_rob)

client_data["ocsvm_label_rob"] = pred

In [ ]:
for pid in FRAUD_IDS:
    clusters = client_data.loc[client_data['person_id'] == pid, 'ocsvm_label_rob'].tolist()
    print(f"person_id={pid} -> clusters={clusters}")


print()
client_data['ocsvm_label_rob'].value_counts()

hit_rate_rob = client_data[client_data['is_fraud']]['ocsvm_label_rob'].value_counts(normalize=True).get(-1, 0)
print(f"Hit Rate (RobustScaler): {hit_rate_rob:.2%}")

person_id=11968000 -> clusters=[-1]  
person_id=11970409 -> clusters=[-1]  
person_id=11726701 -> clusters=[1]  
person_id=14827913 -> clusters=[-1]  
person_id=12411311 -> clusters=[-1]  
person_id=11098795 -> clusters=[]  
person_id=12412748 -> clusters=[-1]  
person_id=11812494 -> clusters=[-1]  
person_id=12396334 -> clusters=[-1]  
  
Hit Rate (RobustScaler): 87.50%  

In [ ]:
client_data["group"] = np.where(client_data["ocsvm_label_rob"] == -1, "Anomaly_Rob", "Normal")

client_data.loc[client_data["person_id"].isin(FRAUD_IDS), "group"] = "Known fraud"
group_order = ["Normal", "Anomaly_Rob", "Known fraud"]

medians_wide = (
    client_data
    .groupby("group")[FEATURES]
    .median()
    .reindex(group_order)
    .reset_index()
)

medians_long = medians_wide.melt(
    id_vars="group",
    value_vars=FEATURES,
    var_name="feature",
    value_name="median_value"
)

fig = px.bar(
    medians_long,
    x="group",
    y="median_value",
    color='group',
    facet_col="feature",
    facet_col_wrap=4,
    category_orders={"group": group_order, "feature": FEATURES},
    title="Feature medians: Normal vs Anomaly vs Known fraud (client_data, standardized with RobustScaler)"
)

fig.update_xaxes(title=None, tickangle=0)
fig.update_yaxes(title="median value")
fig.for_each_annotation(lambda a: a.update(text=a.text.split("=")[-1])) 
fig.update_layout(height=320 * int(np.ceil(len(FEATURES)/4)), showlegend=False)
fig.update_yaxes(matches=None)

fig.show()

In [ ]:
# compare all

client_data["group"] = np.where(client_data["ocsvm_label_rob"] == -1, "Anomaly_Rob", np.where(client_data["ocsvm_label_std"] == -1, "Anomaly_Std", "Normal"))

client_data.loc[client_data["person_id"].isin(FRAUD_IDS), "group"] = "Known fraud"
group_order = ["Normal", "Anomaly_Rob", "Anomaly_Std", "Known fraud"]

medians_wide = (
    client_data
    .groupby("group")[FEATURES]
    .median()
    .reindex(group_order)
    .reset_index()
)

medians_long = medians_wide.melt(
    id_vars="group",
    value_vars=FEATURES,
    var_name="feature",
    value_name="median_value"
)

fig = px.bar(
    medians_long,
    x="group",
    y="median_value",
    color='group',
    facet_col="feature",
    facet_col_wrap=4,
    category_orders={"group": group_order, "feature": FEATURES},
    title="Feature medians: Normal vs Anomaly vs Known fraud (client_data, standardized with RobustScaler and StandardScaler)"
)

fig.update_yaxes(title="median value")
fig.for_each_annotation(lambda a: a.update(text=a.text.split("=")[-1])) 
fig.update_layout(height=320 * int(np.ceil(len(FEATURES)/4)), showlegend=True)
fig.update_yaxes(matches=None)

fig.show()